# Module 3 — Follow-up questions (clarification + session resume)

A good analyst doesn't guess at an ambiguous request — they ask. *"Show me the top students"* —
top by **GPA**? by **credits**? for **which semester**? In this module the agent asks a
**clarification question** before writing SQL, you answer, and it continues — across multiple
rounds if needed.

The twist: AgentCore is **serverless**, so the deployed agent can't pause and wait for terminal
input. Instead it emits its questions as a structured JSON block and returns; a small driver
collects your answers and **re-invokes with session resume** so the agent picks up where it left off.


## It's the same agent — one flag flips it on

No new agent file. `build_agent_options()` (the single source of truth from Module 1) takes an
`enable_clarification` flag:

```python
build_agent_options(request_id=rid, enable_clarification=True, can_use_tool=...)
```

That adds the **AskUserQuestion** tool and a short clarification instruction to the system prompt.
The deploy entrypoint (`agent_agentcore.py`) sets it, plus:
- a `can_use_tool` callback that **allows** AskUserQuestion (it doesn't try to answer it — serverless),
- detects the AskUserQuestion call in the stream → emits a `clarification_needed` JSON block (with the
  SDK **session id**) → returns,
- accepts `claude_agent_sdk_session_id` in the payload to **resume** a prior session.

A drift-guard test confirms this module's `agent.py` is still byte-identical to Module 1's — the
clarification behavior is an *override*, not a fork.

## Setup

Same as Module 2 (AgentCore CLI + CDK + uv env). Run Module 0 first for the data layer.
After the cell below completes, **select the `agentic-analytics-module-3-follow-up` kernel** from
the kernel picker (top-right) and continue with the rest of the notebook.

### Setup step 1

When you run the first script, it will ask you to select a environment

![](images/select-system-python.png)

and you can select the global env for now, and in this case it is 3.11.15 but this version may change

![](images/select-python-global-env.png)

once selected, you can rerun the setup.sh script

In [1]:
!bash setup.sh

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇npm warn deprecated inflight@1.0.6: This module is not supported, and leaks memory. Do not use it. Check out lru-cache if you want a good and tested way to coalesce async requests by a key value, which is much more comprehensive and powerful.
⠇⠏npm warn deprecated glob@7.2.3: Old versions of glob are not supported, and contain widely publicized security vulnerabilities, which have been fixed in the current version. Please update. Support for old versions may be purchased (at exorbitant rates) by contacting i@izs.me
⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 336 packages, and audited 374 packages in 6s
⠼
⠼46 packages are looking for funding
⠼  run `npm fund` for details
⠼
1 moderate severity vulnerability

To address all issues, run:
  npm audit fix

Run `npm audit` for details.
⠼Using CPython 3.11.14
Creating virtual environment at: .venv
Resolved 176 packages in 225ms                                       
Installed 154 packages in 553ms                    

### Setup step 2

Once you see the dependencies and kernel spec `agentic-analytics-module-3-follow-up` are installed per the message from the last step, please refresh your browser (not refresh kernel but browser)

![](images/refresh-browser.png)

and once refreshed, click on the button (it probably shows a python version 3.11.15) you used to select kernel in the preview section

![](images/current-python.png)

it will show you the option to select another kernel and please click

![](images/select-another-kernel.png)

once clicked, you will see the option to select a Jupyter kernel — please click on "Jupyter Kernel"

![](images/select-jupyter-kernel.png)

once clicked, you can see our registered module kernel. The screenshot below shows module-1 as an example, but ***please select `agentic-analytics-module-3-follow-up` since you are working on Module 3***

![](images/example-select-module-1-jupter-kernel.png)

once selected, you will see it as your active kernel. Again the screenshot shows module-1 as an example, ***please select accordingly depending on which module you are working on — for this module, select `agentic-analytics-module-3-follow-up`***

![](images/example-module-1-jupyter-kernel-selected.png)

In [1]:
import os, json
from dotenv import load_dotenv
load_dotenv()
import boto3
acct = boto3.client("sts").get_caller_identity()["Account"]
region = os.getenv("AWS_REGION", "us-west-2")
with open("agentcore/aws-targets.json", "w") as f:
    json.dump([{"name": "default", "account": acct, "region": region}], f, indent=2)
print("target:", acct, region)

target: 490753044219 us-east-1


## Deploy

Same lifecycle as Module 2 — the runtime is configured with `enableOtel` and the Athena/Glue/S3
permissions, and it points at this module's `analytics_agent/` bundle (the clarification-enabled
entrypoint).

In [2]:
!agentcore deploy -y

✓ Load deployment target
⠋ Validate project...(node:358710) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
✓ Validate project
✓ Build CDK project...
✓ Synthesize CloudFormation...
✓ Check bootstrap status...
✓ Check stack status...
⠋ Deploy to AWS...(node:358710) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
✓ Deploy to AWS
✓ Persi

## Try an ambiguous question directly

Invoke once with a deliberately ambiguous prompt. Instead of an answer, you'll get a
`clarification_needed` JSON block — the agent is asking *you* to pick. Note the
`claude_agent_sdk_session_id` in the output; that's what lets the next call resume.

In [4]:
!agentcore invoke '{"prompt": "Show me the top students."}' \
    --runtime analytics --session-id agentic-analytics-m3-demo-session-0001

⠋ Invoking agent...(node:363234) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
nvoking agent...
Claude Agent SDK Session ID: 89446a6c-c30e-4620-a53a-9df25392c6af
The question "top students" is ambiguous — there are several ways to define "top." Let me ask a couple of clarifying questions before I proceed.

```json
{
  "request_id": "3ac78470-356b-4084-8bfb-0b0db6789778",
  "claude_agent_sdk_session_id": "89446a6c-c30e-4620-a53a-9df25392c6af",
  "status": "clarification_needed",
  "questions": [
    {
      "question": "What metric would you like to rank students by?",
      "header": "Ranking by",
      "options": [
      

## Let the driver close the loop (multi-round)

`scripts/invoke_agentcore.py` automates the round-trip: it invokes, detects the
`clarification_needed` block, prompts you for answers in the terminal, then **re-invokes with the
session id** to resume — repeating until the agent gives a final answer.

Because it reads your answers interactively, run it in a **terminal** (not a notebook cell):

```bash
uv run python scripts/invoke_agentcore.py "Show me the top students"
```

You'll see something like:
```
The agent needs clarification:
  Rank by: Which metric should rank "top" students?
    1. GPA — academic standing
    2. Credits earned — progress toward graduation
Your choice (number, comma-separated, or free text): 1
▶ invoking (resume)…
The top 10 students by GPA are…
✅ Done — the agent answered.
```

in the terminal, once running this script, the agent will you clarification question and you can type in your answer and it may look below (screenshots)


![](images/follow-up-question-1.png)

![](images/follow-up-question-2.png)

## See the trace + clean up

The clarification rounds are traced in CloudWatch just like Module 2 (observability is on).

In [ ]:
!agentcore traces list --runtime analytics --since 1h

In [ ]:
# Teardown when finished:
# !aws cloudformation delete-stack --stack-name AgentCore-aafollowup-default --region {region}
print("Uncomment the teardown line above when you're done.")

## Recap

- **Clarification is one flag** on the shared `build_agent_options()` — `enable_clarification=True`
  adds AskUserQuestion. The agent's `agent.py` never forked (drift-guard proves it).
- Serverless multi-round works by **emitting questions as JSON + resuming the SDK session**, driven
  by `scripts/invoke_agentcore.py`.
- Everything else — Athena tool, skills, deploy, observability — is inherited from Modules 1–2.

That completes the Agentic Analytics track: **set up the data (M0) → build the agent (M1) → deploy
& observe it (M2) → make it ask good follow-up questions (M3).**